In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "4" #  set the number of threads for OpenMP and MKL to 4
os.environ["MKL_NUM_THREADS"] = "4" # set the number of threads for OpenMP and MKL to 4

import torch

torch.set_num_threads(4)

In [2]:
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import ultralytics
import yaml
from ultralytics import YOLO

In [3]:
ultralytics.checks()

Ultralytics 8.4.116  Python-3.14.7 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
Setup complete  (16 CPUs, 31.6 GB RAM, 151.3/983.1 GB disk)


In [4]:
cwd = Path.cwd()
p = cwd
while p != p.parent and p.name != "lesson_21":
    p = p.parent

ROOT = p if p.name == "lesson_21" else cwd  # fallback
print("CWD:", cwd)
print("ROOT:", ROOT.resolve())

CWD: C:\Users\Tymur\Projects\ComputerVision\RobotDreams\lesson_21\training\YOLO
ROOT: C:\Users\Tymur\Projects\ComputerVision\RobotDreams\lesson_21


In [5]:
# Make paths robust: notebook is in course_work/training => parent is course_work
# ROOT = Path.cwd().parent

DATASET_SRC = ROOT / "data" / "coin_dataset"
IMAGES_SRC = DATASET_SRC / "images"
LABELS_SRC = DATASET_SRC / "labels"

print("CWD:", Path.cwd())
print("ROOT:", ROOT.resolve())
print("IMAGES_SRC:", IMAGES_SRC.resolve())
print("LABELS_SRC:", LABELS_SRC.resolve())

BASE_MODEL = 'yolo26s.pt'

CWD: C:\Users\Tymur\Projects\ComputerVision\RobotDreams\lesson_21\training\YOLO
ROOT: C:\Users\Tymur\Projects\ComputerVision\RobotDreams\lesson_21
IMAGES_SRC: C:\Users\Tymur\Projects\ComputerVision\RobotDreams\lesson_21\data\coin_dataset\images
LABELS_SRC: C:\Users\Tymur\Projects\ComputerVision\RobotDreams\lesson_21\data\coin_dataset\labels


In [6]:
assert IMAGES_SRC.exists(), IMAGES_SRC
assert LABELS_SRC.exists(), LABELS_SRC

names = ["1 cent","2 cent","5 cent","10 cent","20 cent","50 cent","1 euro","2 euro"]
nc = len(names)

In [7]:
DATASET_SRC

WindowsPath('C:/Users/Tymur/Projects/ComputerVision/RobotDreams/lesson_21/data/coin_dataset')

In [8]:
OUT = ROOT / "data" / "coin_yolo_split"
images_out = OUT / "images"
labels_out = OUT / "labels"

for split in ["train", "val", "test"]:
    (images_out / split).mkdir(parents=True, exist_ok=True)
    (labels_out / split).mkdir(parents=True, exist_ok=True)

# Build pairs
image_files = sorted(IMAGES_SRC.glob("*.jpg"))
pairs = []
for img in image_files:
    lab = LABELS_SRC / f"{img.stem}.txt"
    if lab.exists():
        pairs.append((img, lab))

print("Found pairs:", len(pairs))
assert len(pairs) > 0

# Skip copying if already done
already = list((images_out / "train").glob("*.jpg"))
if len(already) == 0:
    random.seed(42)
    random.shuffle(pairs)
    n = len(pairs)
    n_train = int(0.8 * n)
    n_val = int(0.1 * n)

    train_pairs = pairs[:n_train]
    val_pairs = pairs[n_train:n_train + n_val]
    test_pairs = pairs[n_train + n_val:]

    def copy_pairs(pairs_list, split):
        for img, lab in pairs_list:
            shutil.copy2(img, images_out / split / img.name)
            shutil.copy2(lab, labels_out / split / lab.name)

    copy_pairs(train_pairs, "train")
    copy_pairs(val_pairs, "val")
    copy_pairs(test_pairs, "test")
    print("Copied train/val/test:", len(train_pairs), len(val_pairs), len(test_pairs))
else:
    print("Split already exists. train images:", len(already))



Found pairs: 150
Split already exists. train images: 120


# Create yaml file

In [9]:
data_yaml = {
    "path": str(OUT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": nc,
    "names": names,
}

yaml_path = OUT / "data.yaml"
yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding="utf-8")
print("Wrote:", yaml_path)

Wrote: C:\Users\Tymur\Projects\ComputerVision\RobotDreams\lesson_21\data\coin_yolo_split\data.yaml


# Train

- https://docs.ultralytics.com/modes/train/

In [10]:
device = 0 if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: 0


In [11]:
model = YOLO(BASE_MODEL)

results = model.train(
    data=str(yaml_path),

    # Core training
    epochs=25,              # ↓ from 50 (CPU realistic)
    imgsz=256,              # 🔑 critical balance
    batch=16,

    # Device
    device=device,

    # Stability
    workers=4,
    cache=False,
    amp=True,

    # Augmentations (controlled)
    mosaic=0.0,             # keep OFF (huge RAM spikes)
    mixup=0.0,

    # Light augmentations only
    scale=0.3,
    translate=0.05,
    shear=1.0,
    perspective=0.0,

    hsv_h=0.01,
    hsv_s=0.5,
    hsv_v=0.3,
    fliplr=0.5,

    # Logging
    verbose=True
)

New https://pypi.org/project/ultralytics/8.4.120 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.116  Python-3.14.7 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Tymur\Projects\ComputerVision\RobotDreams\lesson_21\data\coin_yolo_split\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=256, iou=0.7, keras=False, kobj=1.0, line_wid

#  Statistics

In [12]:
run_dir = Path(results.save_dir)

best_pt = run_dir / "weights" / "best.pt"

if best_pt.exists():
    print("Best weights:", best_pt)
else:
    print("Best weights not found. Available files:", list((run_dir / "weights").glob("*")))

Best weights: C:\Users\Tymur\Projects\ComputerVision\runs\detect\train\weights\best.pt


In [13]:
eval_model = YOLO(str(best_pt))
val_metrics = eval_model.val(data=str(yaml_path),
                             split="val",
                             imgsz=512,
                             device=device)
print(val_metrics)

Ultralytics 8.4.116  Python-3.14.7 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
YOLO26s summary (fused): 122 layers, 9,468,276 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 838.8100.1 MB/s, size: 41.3 KB)
val: Scanning C:\Users\Tymur\Projects\ComputerVision\RobotDreams\lesson_21\data\coin_yolo_split\labels\val.cache... 15 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 15/15 9.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 7.7it/s 0.1s
                   all         15         32      0.593      0.325      0.423      0.362
                1 cent          3          3          1      0.399      0.665      0.413
                2 cent          2          2          1          0      0.308      0.294
                5 cent          3          3      0.268      0.333      0.352       0.27
               10 cent          4          4          1         

In [14]:
# Display Ultralytics-saved confusion matrix image (most reliable)
cm_png = run_dir / "confusion_matrix.png"
if cm_png.exists():
    from PIL import Image
    img = Image.open(cm_png)
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix (Ultralytics)")
    plt.show()
else:
    print("confusion_matrix.png not found in:", run_dir)

<Figure size 1000x1000 with 1 Axes>

In [15]:
box = val_metrics.box

# Per-class mAP50-95 is commonly available as `box.maps`
maps = getattr(box, "maps", None)

rows = []
if maps is not None:
    for i, c in enumerate(names):
        rows.append({"class": c, "mAP50-95": float(maps[i])})
    df = pd.DataFrame(rows).sort_values("mAP50-95", ascending=False)
    display(df)
else:
    print("Could not find per-class `maps` on val_metrics.box. Available:", dir(box))

,class,mAP50-95
3,10 cent,0.566393
7,2 euro,0.503221
0,1 cent,0.412786
6,1 euro,0.396643
1,2 cent,0.293571
2,5 cent,0.270009
5,50 cent,0.243041
4,20 cent,0.213257


In [16]:
for k in ["p", "r", "map50", "map"]:
    v = getattr(box, k, None)
    print(k, v)

p [          1           1     0.26795           1     0.21156     0.12659     0.63541     0.50043]
r [    0.39867           0     0.33333           0     0.27354     0.66667         0.5     0.42857]
map50 0.4229998026918257
map 0.3623652554402616


In [17]:
# Evaluate at one image
run_dir = Path(results.save_dir)
best_pt = run_dir / "weights" / "best.pt"

assert best_pt.exists(), best_pt
print("Best weights:", best_pt)

eval_model = YOLO(str(best_pt))

Best weights: C:\Users\Tymur\Projects\ComputerVision\runs\detect\train\weights\best.pt


In [18]:
path = '../../data/coin_dataset/images/018.jpg'

eval_model.predict(source=path,
                   show=True,
                   save=True)


image 1/1 C:\Users\Tymur\Projects\ComputerVision\RobotDreams\lesson_21\training\YOLO\..\..\data\coin_dataset\images\018.jpg: 256x192 1 2 cent, 1 5 cent, 1 1 euro, 5.9ms
Speed: 0.5ms preprocess, 5.9ms inference, 0.3ms postprocess per image at shape (1, 3, 256, 192)
Results saved to C:\Users\Tymur\Projects\ComputerVision\runs\detect\predict


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: None
 names: {0: '1 cent', 1: '2 cent', 2: '5 cent', 3: '10 cent', 4: '20 cent', 5: '50 cent', 6: '1 euro', 7: '2 euro'}
 obb: None
 orig_img: array([[[147, 157, 167],
         [149, 159, 169],
         [150, 160, 170],
         ...,
         [137, 146, 159],
         [137, 146, 159],
         [137, 146, 159]],
 
        [[141, 151, 161],
         [143, 153, 163],
         [143, 153, 163],
         ...,
         [140, 149, 162],
         [140, 149, 162],
         [140, 149, 162]],
 
        [[135, 145, 155],
         [137, 147, 157],
         [137, 147, 157],
         ...,
         [140, 149, 162],
         [140, 149, 162],
         [140, 149, 162]],
 
        ...,
 
        [[132, 144, 156],
         [133, 145, 157],
         [136, 148, 160],
         ...,
         [139, 143, 148],
         [140, 144, 149],
         [142, 146, 151]],
 
   

In [19]:
path = '../../data/test/beach_01.jpg'

eval_model.predict(source=path,
                   show=True,
                   save=True)


image 1/1 C:\Users\Tymur\Projects\ComputerVision\RobotDreams\lesson_21\training\YOLO\..\..\data\test\beach_01.jpg: 256x256 14 20 cents, 6 50 cents, 1 1 euro, 2 2 euros, 36.9ms
Speed: 1.4ms preprocess, 36.9ms inference, 0.4ms postprocess per image at shape (1, 3, 256, 256)
Results saved to C:\Users\Tymur\Projects\ComputerVision\runs\detect\predict


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: None
 names: {0: '1 cent', 1: '2 cent', 2: '5 cent', 3: '10 cent', 4: '20 cent', 5: '50 cent', 6: '1 euro', 7: '2 euro'}
 obb: None
 orig_img: array([[[171, 169, 168],
         [181, 179, 178],
         [192, 190, 189],
         ...,
         [183, 179, 178],
         [178, 174, 173],
         [175, 171, 170]],
 
        [[180, 178, 177],
         [186, 184, 183],
         [194, 192, 191],
         ...,
         [181, 177, 176],
         [178, 174, 173],
         [175, 171, 170]],
 
        [[176, 174, 173],
         [180, 178, 177],
         [187, 185, 184],
         ...,
         [181, 177, 176],
         [179, 175, 174],
         [176, 172, 171]],
 
        ...,
 
        [[180, 184, 189],
         [175, 179, 184],
         [177, 181, 186],
         ...,
         [183, 187, 188],
         [182, 186, 187],
         [172, 176, 177]],
 
   